In [ ]:
# ─────────────────────────────────────────────────
# VERIFY ENVIRONMENT
# ─────────────────────────────────────────────────

import transformers
import datasets
import seqeval
import evaluate
import accelerate
import torch

print("=" * 50)
print("✅ transformers :", transformers.__version__)
print("✅ datasets     :", datasets.__version__)
print("✅ evaluate     :", evaluate.__version__)
print("✅ accelerate   :", accelerate.__version__)
print("✅ torch        :", torch.__version__)
print("✅ seqeval      : loaded successfully")
print("=" * 50)

print(f"🖥️ GPU Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🖥️ GPU Name     : {torch.cuda.get_device_name(0)}")
print("=" * 50)

✅ transformers : 5.5.0
✅ datasets     : 4.8.4
✅ evaluate     : 0.4.6
✅ accelerate   : 1.13.0
✅ torch        : 2.10.0+cu128
✅ seqeval      : loaded successfully
🖥️ GPU Available : True
🖥️ GPU Name     : Tesla T4


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from google.colab import files
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset
import evaluate
from seqeval.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

print("✅ All imports successful!")
print(f"🖥️  GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND'}")

✅ All imports successful!
🖥️  GPU : Tesla T4


In [ ]:
print("⬆️  Upload your 3 files: eng.train, eng.testa, eng.tstb")
uploaded = files.upload()
print(f"\n✅ Files uploaded: {list(uploaded.keys())}")

⬆️  Upload your 3 files: eng.train, eng.testa, eng.tstb


Saving eng.train to eng (2).train
Saving eng.testa to eng (3).testa
Saving eng.testb to eng (2).testb

✅ Files uploaded: ['eng (2).train', 'eng (3).testa', 'eng (2).testb']


In [ ]:
def parse_conll_file(filepath):
    """
    Reads CoNLL-2003 file.
    Each line: WORD  POS-TAG  CHUNK-TAG  NER-TAG
    Blank lines = sentence boundary
    """
    sentences = []
    words, pos_tags, chunk_tags = [], [], []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if line.startswith('-DOCSTART-'):
                continue

            if line == '':
                if words:
                    sentences.append({
                        'tokens'     : words,
                        'pos_tags'   : pos_tags,
                        'chunk_tags' : chunk_tags
                    })
                    words, pos_tags, chunk_tags = [], [], []
            else:
                parts = line.split()
                if len(parts) >= 3:
                    words.append(parts[0])
                    pos_tags.append(parts[1])
                    chunk_tags.append(parts[2])

    # Catch last sentence
    if words:
        sentences.append({
            'tokens'     : words,
            'pos_tags'   : pos_tags,
            'chunk_tags' : chunk_tags
        })

    return sentences


# ── Auto-detect test filename ──
test_filename = 'eng.tstb' if os.path.exists('eng.tstb') else 'eng.testb'

train_data = parse_conll_file('eng.train')
val_data   = parse_conll_file('eng.testa')
test_data  = parse_conll_file(test_filename)

print(f"✅ Train      : {len(train_data):,} sentences")
print(f"✅ Validation : {len(val_data):,} sentences")
print(f"✅ Test       : {len(test_data):,} sentences")

# ── Preview sample ──
s = train_data[2]
print(f"\n📄 Sample Preview:")
print(f"  {'Word':<18} {'POS':<10} {'Chunk':<12}")
print(f"  {'─'*18} {'─'*10} {'─'*12}")
for w, p, c in zip(s['tokens'][:8], s['pos_tags'][:8], s['chunk_tags'][:8]):
    print(f"  {w:<18} {p:<10} {c:<12}")

✅ Train      : 14,041 sentences
✅ Validation : 3,250 sentences
✅ Test       : 3,453 sentences

📄 Sample Preview:
  Word               POS        Chunk       
  ────────────────── ────────── ────────────
  BRUSSELS           NNP        B-NP        
  1996-08-22         CD         I-NP        


In [ ]:
# ── Collect all unique labels from training data ──
all_chunk_labels = sorted(set(t for s in train_data for t in s['chunk_tags']))
all_pos_labels   = sorted(set(t for s in train_data for t in s['pos_tags']))

# ── Choose task: 'chunk' or 'pos' ──
TASK = 'chunk'

if TASK == 'chunk':
    all_labels = all_chunk_labels
    label_key  = 'chunk_tags'
else:
    all_labels = all_pos_labels
    label_key  = 'pos_tags'

label2id   = {label: idx for idx, label in enumerate(all_labels)}
id2label   = {idx: label for label, idx in label2id.items()}
NUM_LABELS = len(label2id)

print(f"✅ Task       : {TASK.upper()} Tagging")
print(f"✅ Labels     : {NUM_LABELS}")
print(f"✅ Label list : {all_labels}")

✅ Task       : CHUNK Tagging
✅ Labels     : 20
✅ Label list : ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-SBAR', 'I-VP', 'O']


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded: {MODEL_NAME}")

✅ Tokenizer loaded: distilbert-base-uncased


In [ ]:
# ─────────────────────────────────────────────────
# CELL 7 — FIXED: Tokenize & Align Labels
# Fix: Build label2id from ALL splits (train+val+test)
# so no unknown label causes a KeyError
# ─────────────────────────────────────────────────

# ── Rebuild label mappings from ALL data ──
all_data = train_data + val_data + test_data

all_chunk_labels = sorted(set(t for s in all_data for t in s['chunk_tags']))
all_pos_labels   = sorted(set(t for s in all_data for t in s['pos_tags']))

# Apply to chosen task
if TASK == 'chunk':
    all_labels = all_chunk_labels
    label_key  = 'chunk_tags'
else:
    all_labels = all_pos_labels
    label_key  = 'pos_tags'

label2id   = {label: idx for idx, label in enumerate(all_labels)}
id2label   = {idx: label for label, idx in label2id.items()}
NUM_LABELS = len(label2id)

print(f"✅ Labels rebuilt from ALL splits : {NUM_LABELS} labels")
print(f"✅ Labels : {all_labels}")


# ── Tokenize & Align ──
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation          = True,
        is_split_into_words = True,
        max_length          = 128,
        padding             = False
    )

    all_labels_out = []
    for i, labels in enumerate(examples[label_key]):
        word_ids      = tokenized_inputs.word_ids(batch_index=i)
        previous_word = None
        label_ids     = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word:
                # ── Safe lookup with fallback ──
                tag = labels[word_idx]
                label_ids.append(label2id.get(tag, -100))  # -100 if unseen tag
            else:
                label_ids.append(-100)
            previous_word = word_idx

        all_labels_out.append(label_ids)

    tokenized_inputs['labels'] = all_labels_out
    return tokenized_inputs


# ── Convert to HuggingFace Dataset ──
def to_hf_dataset(data):
    return Dataset.from_dict({
        'tokens'     : [s['tokens']     for s in data],
        'pos_tags'   : [s['pos_tags']   for s in data],
        'chunk_tags' : [s['chunk_tags'] for s in data],
    })

remove_cols = ['tokens', 'pos_tags', 'chunk_tags']

train_tokenized = to_hf_dataset(train_data).map(
    tokenize_and_align_labels, batched=True, remove_columns=remove_cols)
val_tokenized   = to_hf_dataset(val_data).map(
    tokenize_and_align_labels, batched=True, remove_columns=remove_cols)
test_tokenized  = to_hf_dataset(test_data).map(
    tokenize_and_align_labels, batched=True, remove_columns=remove_cols)

print(f"\n✅ Train tokenized : {len(train_tokenized)}")
print(f"✅ Val   tokenized : {len(val_tokenized)}")
print(f"✅ Test  tokenized : {len(test_tokenized)}")

# ── Verify sample ──
sample = train_tokenized[0]
print(f"\n📌 input_ids      : {sample['input_ids'][:8]}")
print(f"📌 attention_mask : {sample['attention_mask'][:8]}")
print(f"📌 labels         : {sample['labels'][:8]}")

✅ Labels rebuilt from ALL splits : 21 labels
✅ Labels : ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-VP', 'O']


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]


✅ Train tokenized : 14041
✅ Val   tokenized : 3250
✅ Test  tokenized : 3453

📌 input_ids      : [101, 7327, 19164, 2446, 2655, 2000, 17757, 2329]
📌 attention_mask : [1, 1, 1, 1, 1, 1, 1, 1]
📌 labels         : [-100, 5, 9, 5, 15, 9, 19, 5]


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels              = NUM_LABELS,
    id2label                = id2label,
    label2id                = label2id,
    ignore_mismatched_sizes = True
)

print(f"✅ Model loaded     : {MODEL_NAME}")
print(f"✅ Total parameters : {model.num_parameters():,}")
print(f"✅ Output labels    : {NUM_LABELS}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded     : distilbert-base-uncased
✅ Total parameters : 66,379,029
✅ Output labels    : 21


In [ ]:
data_collator  = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(p):
    logits, labels = p
    predictions    = np.argmax(logits, axis=2)

    true_predictions = [
        [id2label[int(pred)] for pred, lbl in zip(row_p, row_l) if int(lbl) != -100]
        for row_p, row_l in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[int(lbl)] for pred, lbl in zip(row_p, row_l) if int(lbl) != -100]
        for row_p, row_l in zip(predictions, labels)
    ]

    results = seqeval_metric.compute(
        predictions = true_predictions,
        references  = true_labels
    )
    return {
        'precision' : round(results['overall_precision'], 4),
        'recall'    : round(results['overall_recall'],    4),
        'f1'        : round(results['overall_f1'],        4),
    }

print("✅ Data collator and metrics ready!")

✅ Data collator and metrics ready!


In [ ]:

training_args = TrainingArguments(
    output_dir                  = './results',
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    logging_steps               = 200,
    report_to                   = 'none',
    fp16                        = True,
)

trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_tokenized,
    eval_dataset     = val_tokenized,
    processing_class = tokenizer,        # ✅ fixed (was tokenizer=)
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
)

print("🚀 Training started...")
train_result = trainer.train()
print(f"\n✅ Training complete!")
print(f"   Loss  : {train_result.training_loss:.4f}")
print(f"   Steps : {train_result.global_step}")

🚀 Training started...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.209123,0.199867,0.904700,0.900600,0.902600
2,0.155407,0.177647,0.913500,0.909400,0.911500
3,0.130903,0.172514,0.916300,0.911100,0.913700


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ Training complete!
   Loss  : 0.2180
   Steps : 2634


In [ ]:
from transformers import pipeline
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score
import numpy as np

# ─────────────────────────────────────────────────
# CELL 11 — FIXED EVALUATION (no Trainer callback bug)
# Uses model.forward() directly — avoids the
# "on_train_begin must be called" error completely
# ─────────────────────────────────────────────────

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

def evaluate_dataset(tokenized_dataset):
    """
    Runs inference directly using model — no Trainer needed.
    Returns aligned true labels and predictions.
    """
    from torch.utils.data import DataLoader

    # Use data_collator for proper padding
    dataloader = DataLoader(
        tokenized_dataset,
        batch_size    = 32,
        collate_fn    = data_collator
    )

    all_preds  = []
    all_labels = []

    for batch in dataloader:
        # Move batch to GPU
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels']             # Keep on CPU

        with torch.no_grad():
            outputs = model(
                input_ids      = input_ids,
                attention_mask = attention_mask
            )

        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        labels = labels.numpy()

        # Align — skip -100 positions
        for row_pred, row_label in zip(preds, labels):
            sp, sl = [], []
            for p, l in zip(row_pred, row_label):
                if int(l) != -100:
                    sp.append(id2label[int(p)])
                    sl.append(id2label[int(l)])
            all_preds.append(sp)
            all_labels.append(sl)

    return all_labels, all_preds


# ── Evaluate Validation Set ──
print("=" * 50)
print("📊 EVALUATION RESULTS")
print("=" * 50)

print("\n⏳ Evaluating Validation Set...")
val_true, val_pred = evaluate_dataset(val_tokenized)
print(f"\n🔹 Validation Set:")
print(f"   Precision : {precision_score(val_true, val_pred):.4f}")
print(f"   Recall    : {recall_score(val_true,    val_pred):.4f}")
print(f"   F1 Score  : {f1_score(val_true,        val_pred):.4f}")

# ── Evaluate Test Set ──
print("\n⏳ Evaluating Test Set...")
test_true, test_pred = evaluate_dataset(test_tokenized)
print(f"\n🔹 Test Set:")
print(f"   Precision : {precision_score(test_true, test_pred):.4f}")
print(f"   Recall    : {recall_score(test_true,    test_pred):.4f}")
print(f"   F1 Score  : {f1_score(test_true,        test_pred):.4f}")

# ── Per-Class Report ──
print("\n" + "=" * 50)
print("📋 PER-CLASS REPORT (Test Set)")
print("=" * 50)
print(classification_report(test_true, test_pred, digits=4))

# ── Save results for Report cell later ──
test_results = {
    'eval_precision' : precision_score(test_true, test_pred),
    'eval_recall'    : recall_score(test_true,    test_pred),
    'eval_f1'        : f1_score(test_true,        test_pred),
}
print("✅ Evaluation complete! Results saved to test_results dict.")

📊 EVALUATION RESULTS

⏳ Evaluating Validation Set...

🔹 Validation Set:
   Precision : 0.9163
   Recall    : 0.9111
   F1 Score  : 0.9137

⏳ Evaluating Test Set...

🔹 Test Set:
   Precision : 0.9082
   Recall    : 0.8983
   F1 Score  : 0.9032

📋 PER-CLASS REPORT (Test Set)
              precision    recall  f1-score   support

        ADJP     0.7033    0.6268    0.6628       276
        ADVP     0.7816    0.7299    0.7549       559
       CONJP     0.0000    0.0000    0.0000         6
        INTJ     0.0000    0.0000    0.0000        13
         LST     0.0000    0.0000    0.0000        29
          NP     0.9019    0.8968    0.8994     12983
          PP     0.9654    0.9812    0.9732      3979
         PRT     0.7647    0.7091    0.7358       110
        SBAR     0.8643    0.8176    0.8403       296
          VP     0.9069    0.8843    0.8954      3767

   micro avg     0.9082    0.8983    0.9032     22018
   macro avg     0.5888    0.5646    0.5762     22018
weighted avg     0.905

In [ ]:
def predict_tags(sentence):
    words  = sentence.split()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    model.eval()

    encoding = tokenizer(
        words,
        is_split_into_words = True,
        return_tensors      = 'pt',
        truncation          = True,
        max_length          = 128
    )
    inputs = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        logits   = model(**inputs).logits[0]
    pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
    word_ids = encoding.word_ids()

    results, seen = [], set()
    for idx, w_id in enumerate(word_ids):
        if w_id is not None and w_id not in seen:
            seen.add(w_id)
            results.append((words[w_id], id2label[int(pred_ids[idx])]))
    return results


# ── Test on custom sentences ──
test_sentences = [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "Apple announced a new iPhone model last week",
]

print("=" * 50)
print(f"🔮 INFERENCE — {TASK.upper()} TAGGING")
print("=" * 50)

for sent in test_sentences:
    print(f"\n📝 Input : {sent}")
    print(f"  {'Word':<22} {'Tag':<15}")
    print(f"  {'─'*22} {'─'*15}")
    for word, tag in predict_tags(sent):
        print(f"  {word:<22} {tag:<15}")

🔮 INFERENCE — CHUNK TAGGING

📝 Input : John works at Google in California
  Word                   Tag            
  ────────────────────── ───────────────
  John                   B-NP           
  works                  B-VP           
  at                     B-PP           
  Google                 B-NP           
  in                     B-PP           
  California             B-NP           

📝 Input : The quick brown fox jumps over the lazy dog
  Word                   Tag            
  ────────────────────── ───────────────
  The                    B-NP           
  quick                  I-NP           
  brown                  I-NP           
  fox                    I-NP           
  jumps                  B-VP           
  over                   B-PP           
  the                    B-NP           
  lazy                   I-NP           
  dog                    I-NP           

📝 Input : Apple announced a new iPhone model last week
  Word                   Tag        

In [ ]:
print("=" * 60)
print("⚖️  COMPARISON — POS Tagging vs Chunking")
print("=" * 60)

s = train_data[3]
print(f"\n📝 Sentence: {' '.join(s['tokens'][:10])}\n")
print(f"  {'Word':<18} {'POS Tag':<12} {'Chunk Tag'}")
print(f"  {'─'*18} {'─'*12} {'─'*12}")
for w, p, c in zip(s['tokens'][:10], s['pos_tags'][:10], s['chunk_tags'][:10]):
    print(f"  {w:<18} {p:<12} {c}")

df = pd.DataFrame({
    'Aspect'      : ['Goal', 'Unit', 'Tags', 'Difficulty', 'Example', 'Use Case'],
    'POS Tagging' : ['Word grammar label', 'Single word', 'NN, VB, JJ...', 'Easier', '"runs" → VBZ', 'Grammar check'],
    'Chunking'    : ['Phrase grouping', 'Multiple words', 'B-NP, I-NP, B-VP...', 'Medium', '"the cat" → B-NP I-NP', 'Info extraction'],
})
print(f"\n{df.to_string(index=False)}")

⚖️  COMPARISON — POS Tagging vs Chunking

📝 Sentence: The European Commission said on Thursday it disagreed with German

  Word               POS Tag      Chunk Tag
  ────────────────── ──────────── ────────────
  The                DT           B-NP
  European           NNP          I-NP
  Commission         NNP          I-NP
  said               VBD          B-VP
  on                 IN           B-PP
  Thursday           NNP          B-NP
  it                 PRP          B-NP
  disagreed          VBD          B-VP
  with               IN           B-PP
  German             JJ           B-NP

    Aspect        POS Tagging              Chunking
      Goal Word grammar label       Phrase grouping
      Unit        Single word        Multiple words
      Tags      NN, VB, JJ...   B-NP, I-NP, B-VP...
Difficulty             Easier                Medium
   Example       "runs" → VBZ "the cat" → B-NP I-NP
  Use Case      Grammar check       Info extraction


In [ ]:
print(f"""
{'='*60}
📝  REPORT — Fine-Tuning DistilBERT for Token Classification
{'='*60}

1. DATASET
   CoNLL-2003 | Train: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_data):,}
   Task: {TASK.upper()} Tagging | Labels: {NUM_LABELS}

2. POS TAGGING vs CHUNKING
   • POS Tagging → labels each word's grammatical role (NN, VB…)
   • Chunking    → groups words into phrases (B-NP, I-NP, B-VP…)
   • POS is word-level; Chunking is phrase-level (harder)

3. CHALLENGES
   • Subword tokens: BERT splits words — only first subword
     gets the real label, rest get -100
   • Special tokens [CLS][SEP] also get -100
   • seqeval is strict — entire phrase must match exactly

4. RESULTS
   • Precision : {test_results.get('eval_precision', 'N/A')}
   • Recall    : {test_results.get('eval_recall',    'N/A')}
   • F1 Score  : {test_results.get('eval_f1',        'N/A')}

5. OBSERVATIONS
   • DistilBERT is 40% smaller, 60% faster than BERT
   • 2e-5 learning rate + 3 epochs = stable convergence
   • BIO format helps detect phrase boundaries accurately

{'='*60}
""")


📝  REPORT — Fine-Tuning DistilBERT for Token Classification

1. DATASET
   CoNLL-2003 | Train: 14,041 | Val: 3,250 | Test: 3,453
   Task: CHUNK Tagging | Labels: 21

2. POS TAGGING vs CHUNKING
   • POS Tagging → labels each word's grammatical role (NN, VB…)
   • Chunking    → groups words into phrases (B-NP, I-NP, B-VP…)
   • POS is word-level; Chunking is phrase-level (harder)

3. CHALLENGES
   • Subword tokens: BERT splits words — only first subword
     gets the real label, rest get -100
   • Special tokens [CLS][SEP] also get -100
   • seqeval is strict — entire phrase must match exactly

4. RESULTS
   • Precision : 0.9081684191193351
   • Recall    : 0.8983104732491598
   • F1 Score  : 0.9032125488047127

5. OBSERVATIONS
   • DistilBERT is 40% smaller, 60% faster than BERT
   • 2e-5 learning rate + 3 epochs = stable convergence
   • BIO format helps detect phrase boundaries accurately




In [ ]:
model.save_pretrained('./saved_model')
tokenizer.save_pretrained('./saved_model')
print("✅ Model saved!")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved!


In [54]:
# Run this as the LAST cell before downloading
import json

# Force-save all outputs
print("✅ All cells executed successfully!")
print(f"   Task        : {TASK.upper()} Tagging")
print(f"   Model       : {MODEL_NAME}")
print(f"   Train size  : {len(train_data):,} sentences")
print(f"   Test F1     : {test_results['eval_f1']:.4f}")
print(f"   Precision   : {test_results['eval_precision']:.4f}")
print(f"   Recall      : {test_results['eval_recall']:.4f}")
print(f"   Labels      : {NUM_LABELS}")
print(f"   Labels list : {all_labels}")

✅ All cells executed successfully!
   Task        : CHUNK Tagging
   Model       : distilbert-base-uncased
   Train size  : 14,041 sentences
   Test F1     : 0.9032
   Precision   : 0.9082
   Recall      : 0.8983
   Labels      : 21
   Labels list : ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-VP', 'O']
